# Task 3: Model Training
## BioBERT Pressure Ulcer QA System
## 7146COMP Advanced Topics in Deep Learning

---

## Overview

This notebook implements the fine-tuning of BioBERT Large on the 
pressure ulcer QA dataset constructed in Task 1 and preprocessed 
in Task 2. The fine-tuned model is saved to disk for inference 
and evaluation in Task 4.

---

## Task 3 Design Decisions and Justifications

### Decision 1: BioBERT Large over Standard BERT and BioBERT Base

BioBERT Large (dmis-lab/biobert-large-cased-v1.1-squad) is selected 
as the base model in preference to both standard BERT and BioBERT 
Base for four reasons.

First, BioBERT was pre-trained on 18 billion tokens of biomedical 
literature from PubMed abstracts and full-text articles, meaning 
its token representations are calibrated to medical language from 
the outset. This domain-adapted pre-training is essential for a 
clinical QA system where terms such as debridement, eschar, and 
granulation must be correctly represented.

Second, the Large variant comprises 24 transformer layers and 
approximately 340 million parameters compared to 12 layers and 
110 million parameters in the Base variant. This increased model 
capacity enables richer contextual representations, which is 
particularly beneficial for long clinical passages where answer 
spans may depend on distant contextual cues.

Third, the selected checkpoint has been further fine-tuned on 
the Stanford Question Answering Dataset, providing a strong 
extractive QA foundation before domain-specific fine-tuning on 
the pressure ulcer dataset. This two-stage fine-tuning approach 
— general QA then domain-specific QA — reduces the training 
burden and improves convergence compared to fine-tuning from 
the pre-trained language model alone.

Fourth, the model is available in the local HuggingFace cache, 
eliminating network dependency during training and ensuring 
full reproducibility of results regardless of external service 
availability.

Generative alternatives such as GPT-based models were considered 
but rejected because they are not constrained to return text from 
verified source documents, introducing hallucination risk that is 
unacceptable in a patient safety context.

### Decision 2: AdamW Optimiser with Linear Warmup Schedule

AdamW is selected as the optimiser in preference to standard Adam 
and SGD. Standard Adam incorporates L2 regularisation by adding 
weight decay directly to the gradient update, which couples 
regularisation with the adaptive learning rate mechanism and 
produces suboptimal regularisation for transformer models. AdamW 
decouples weight decay from the gradient update, applying it 
directly to the weights rather than the gradients. Loshchilov and 
Hutter demonstrated that this decoupled approach produces more 
effective regularisation and better generalisation for transformer 
fine-tuning than standard Adam.

A linear warmup schedule is applied over the first 10% of total 
training steps. During the warmup period the learning rate 
increases linearly from near zero to the target value of 2e-5, 
then decays linearly to zero over the remaining training steps. 
The warmup mechanism protects the pre-trained BioBERT weights 
from destabilisation during early training steps when gradient 
signals are most volatile and before the optimiser has sufficient 
exposure to the training distribution to produce reliable updates.

### Decision 3: Learning Rate of 2e-5

A learning rate of 2e-5 is selected at the lower end of the 
range recommended by Devlin et al. for BERT fine-tuning. This 
conservative choice is deliberate because the relatively small 
training corpus of 2,365 examples makes the model vulnerable to 
catastrophic forgetting — noisy gradient updates derived from 
limited examples risk overwriting the domain-adapted biomedical 
knowledge established during BioBERT pre-training. A higher 
learning rate would converge faster but at the cost of 
destabilising the pre-trained representations that make BioBERT 
superior to standard BERT for this task.

### Decision 4: Three Training Epochs with Early Stopping

Training is limited to a maximum of three epochs consistent with 
established BioBERT fine-tuning practice. Additional epochs 
increase the risk of overfitting on a dataset of this size — 
the model adapts too closely to the training distribution and 
loses the ability to generalise to the diverse clinical questions 
a deployed system would encounter.

Early stopping with a patience of two epochs is applied based 
on validation loss. If validation loss fails to improve for two 
consecutive epochs training halts and the best checkpoint is 
restored. This prevents unnecessary computation and ensures the 
saved model represents the optimal point on the training 
trajectory rather than an overfit later checkpoint. Validation 
loss and exact match are monitored after each epoch to track 
both the primary training objective and the downstream task 
performance simultaneously.

### Decision 5: Batch Size of 16

A batch size of 16 is selected to balance gradient stability 
against the memory constraints of the available hardware. Larger 
batches produce more stable gradient estimates but require 
proportionally more GPU memory — at 512 tokens per sequence with 
BioBERT Large's 340 million parameters, a batch size of 32 risks 
exceeding the 25.8 GB VRAM available on the RTX 3090. A batch 
size of 16 is the standard recommendation for BERT fine-tuning 
on single GPU hardware and has been consistently validated across 
domain-specific fine-tuning studies.

## 3.1 Environment Setup and Library Imports

All libraries required for BioBERT fine-tuning are imported in 
this cell. The transformers library provides the AutoModelForQuestionAnswering 
class which loads the BioBERT checkpoint with a question answering 
head — two linear classifiers appended to the final transformer 
layer predicting answer start and end token positions respectively. 
The get_linear_schedule_with_warmup function provides the linear 
warmup and decay schedule described in the design decisions above.

GPU availability is confirmed and the device is set before any 
model or tensor operations. All model parameters and input tensors 
are moved to the GPU device to ensure training runs on the RTX 3090 
rather than CPU, which would increase training time by approximately 
two orders of magnitude.

A fixed random seed is set across Python, NumPy, and PyTorch to 
ensure reproducibility of weight initialisation and data shuffling 
across training runs.

In [1]:
# =============================================================================
# Environment Setup and Library Imports
# =============================================================================

import os
import json
import time
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from tqdm import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Task 3: Model Training")
print(f"Device:     {device}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch:    {torch.__version__}")
print(f"Seed:       {SEED}")
print(f"Libraries imported successfully.")

C:\Users\MSC1\anaconda3\envs\bertqa2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Task 3: Model Training
Device:     cuda
GPU:        NVIDIA GeForce RTX 3090
VRAM:       25.8 GB
PyTorch:    2.5.1+cu121
Seed:       42
Libraries imported successfully.


## 3.2 Configuration

All training hyperparameters are defined in a single configuration 
block. The values are consistent with the design decisions described 
in Section 1 and with the preprocessing configuration in Task 2. 
Centralising all hyperparameters here ensures they can be reviewed 
and adjusted in one place without searching through the training 
loop code.

The model checkpoint path defines where the fine-tuned model will 
be saved after training. The best checkpoint — the epoch with the 
lowest validation loss — is saved separately from the final epoch 
checkpoint, ensuring the optimal model state is preserved even if 
early stopping halts training at a later, overfit epoch.

In [2]:
# =============================================================================
# Configuration
# All training hyperparameters defined in one place.
# Consistent with Task 2 preprocessing configuration.
# =============================================================================

# File paths — Task 2 outputs
PREPROCESSED_DIR = "./preprocessed"
TRAIN_FILE       = os.path.join(PREPROCESSED_DIR, "train_dataset.pt")
VAL_FILE         = os.path.join(PREPROCESSED_DIR, "val_dataset.pt")
TEST_FILE        = os.path.join(PREPROCESSED_DIR, "test_dataset.pt")

# Model
MODEL_NAME       = "dmis-lab/biobert-large-cased-v1.1-squad"
MODEL_SAVE_DIR   = "./biobert_pressure_ulcer"
BEST_MODEL_DIR   = "./biobert_pressure_ulcer_best"

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

# Training hyperparameters
BATCH_SIZE       = 16
LEARNING_RATE    = 2e-5
NUM_EPOCHS       = 3
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.10     # 10% of total steps
EARLY_STOP_PAT   = 2        # Stop if no improvement for 2 epochs
MAX_GRAD_NORM    = 1.0      # Gradient clipping threshold

# Verify preprocessed datasets exist
all_present = True
for name, path in [("Train",      TRAIN_FILE),
                    ("Validation", VAL_FILE),
                    ("Test",       TEST_FILE)]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  ✅ {name:<15} {path} ({size_mb:.1f} MB)")
    else:
        print(f"  ❌ {name:<15} {path} NOT FOUND")
        all_present = False

if all_present:
    print(f"\nHyperparameter Configuration:")
    print(f"  Model:           {MODEL_NAME}")
    print(f"  Batch size:      {BATCH_SIZE}")
    print(f"  Learning rate:   {LEARNING_RATE}")
    print(f"  Epochs:          {NUM_EPOCHS}")
    print(f"  Weight decay:    {WEIGHT_DECAY}")
    print(f"  Warmup ratio:    {WARMUP_RATIO}")
    print(f"  Early stopping:  patience={EARLY_STOP_PAT}")
    print(f"  Grad clip norm:  {MAX_GRAD_NORM}")
else:
    print("\nERROR: Run Task 2 before proceeding.")

  ✅ Train           ./preprocessed\train_dataset.pt (31.3 MB)
  ✅ Validation      ./preprocessed\val_dataset.pt (6.7 MB)
  ✅ Test            ./preprocessed\test_dataset.pt (6.7 MB)

Hyperparameter Configuration:
  Model:           dmis-lab/biobert-large-cased-v1.1-squad
  Batch size:      16
  Learning rate:   2e-05
  Epochs:          3
  Weight decay:    0.01
  Warmup ratio:    0.1
  Early stopping:  patience=2
  Grad clip norm:  1.0


## 3.3 Hyperparameter Justification

The hyperparameters configured in Section 3.2 are not arbitrary 
selections — each value is justified by the characteristics of 
the pressure ulcer dataset, the BioBERT architecture, and 
established fine-tuning practice from the literature.

### Learning Rate: 2e-5

A learning rate of 2e-5 is selected at the lower end of the 
2e-5 to 5e-5 range recommended by Devlin et al. for BERT 
fine-tuning on downstream tasks. The conservative selection 
is motivated by two considerations specific to this dataset. 
First, the training corpus of 2,365 examples is relatively 
small by transformer fine-tuning standards. Higher learning 
rates produce larger gradient steps that risk destabilising 
the pre-trained BioBERT weights, erasing the domain-adapted 
biomedical representations that make BioBERT superior to 
standard BERT for clinical text. This phenomenon, known as 
catastrophic forgetting, is particularly pronounced when the 
fine-tuning corpus is small relative to the pre-training 
corpus. Second, the BioBERT Large checkpoint has already been 
fine-tuned on SQuAD — it begins training in a strong QA 
configuration that requires only modest adjustment to the 
pressure ulcer domain, not a large-scale weight update.

### Batch Size: 16

A batch size of 16 is selected to balance gradient stability 
against GPU memory constraints. Each training example occupies 
a 512-token sequence — at BioBERT Large's 340 million 
parameters, a batch size of 32 at float32 precision risks 
exhausting the 25.8 GB VRAM on the RTX 3090 when accounting 
for activations, gradients, and optimiser states. A batch size 
of 16 is the standard recommendation for single-GPU BERT 
fine-tuning and has been validated in the original Devlin et 
al. paper and in subsequent domain-specific fine-tuning studies. 
Larger batches produce more stable gradient estimates but the 
marginal stability gain from increasing from 16 to 32 does not 
justify the memory risk at this scale.

### Number of Epochs: 3

Training is limited to a maximum of 3 epochs consistent with 
the recommendation of Devlin et al. for BERT fine-tuning. 
Transformer models are highly expressive and begin to overfit 
on small datasets after relatively few passes. With 2,365 
training examples, the model sees each example only 3 times 
before training ends, which is sufficient to adapt the QA 
head and adjust the upper transformer layers to the pressure 
ulcer domain without catastrophic forgetting of the general 
biomedical representations in the lower layers. Early stopping 
with patience of 2 epochs provides an additional safeguard — 
if validation loss degrades before epoch 3, the best checkpoint 
is preserved automatically.

### Weight Decay: 0.01

Weight decay of 0.01 is applied as a regularisation measure 
throughout training. This value penalises large parameter 
values, preventing individual weights from growing excessively 
in response to idiosyncratic training examples. It is applied 
through the AdamW optimiser's decoupled weight decay mechanism 
rather than L2 regularisation, which Loshchilov and Hutter 
demonstrated produces more effective regularisation for 
transformer models. The value of 0.01 is deliberately light 
— heavier regularisation would slow convergence on an already 
small dataset.

### Warmup Ratio: 0.10

A linear warmup over the first 10% of total training steps 
initialises the learning rate at near zero and gradually 
increases it to the target 2e-5. This warmup period allows 
the optimiser to accumulate reliable gradient statistics 
before applying the full learning rate, protecting the 
pre-trained BioBERT weights from large destabilising updates 
in the first few batches when the QA head weights are still 
random and producing high-variance gradients.

### Gradient Clipping: 1.0

Gradient clipping at a maximum L2 norm of 1.0 is applied 
before each optimiser step. This prevents exploding gradients 
— occasionally a batch with unusual characteristics produces 
very large gradient values that would produce an excessively 
large weight update. Clipping rescales the gradient vector 
to a maximum norm of 1.0 without changing its direction, 
ensuring stable training without interfering with the 
learning dynamics in normal batches. This is standard 
practice for transformer fine-tuning.

### Early Stopping: Patience 2

Early stopping with a patience of 2 epochs monitors validation 
loss after each epoch. If validation loss fails to improve for 
2 consecutive epochs training halts and the best checkpoint 
is restored. This prevents wasted computation on epochs that 
are producing overfitting rather than generalisation 
improvement, and ensures the saved model represents the 
optimal point on the training trajectory.

## 3.4 Model and Data Loading

The BioBERT Large model is loaded from the local HuggingFace cache 
with a question answering head. The AutoModelForQuestionAnswering 
class appends two linear classifiers to the final transformer layer 
— one predicting the start token position of the answer span and 
one predicting the end token position. These classifiers are 
randomly initialised and trained from scratch during fine-tuning 
while the 24 transformer layers inherit their weights from the 
BioBERT pre-trained checkpoint.

The model is immediately moved to the GPU device after loading. 
BioBERT Large with 340 million parameters requires approximately 
1.3 GB of GPU memory in float32 precision. With a batch size of 
16 and sequence length of 512 the total GPU memory requirement 
including activations and gradients is estimated at approximately 
8 to 12 GB, well within the 25.8 GB available on the RTX 3090.

The preprocessed PyTorch datasets from Task 2 are loaded directly 
using torch.load. The training DataLoader is configured with 
shuffle=True to randomise example order between epochs, which 
reduces the risk of the model overfitting to the order in which 
examples are presented. The validation and test DataLoaders are 
configured with shuffle=False to ensure evaluation results are 
reproducible across runs.

In [4]:
# =============================================================================
# Dataset Class Definition
# PressureUlcerQADataset must be defined in this notebook before
# loading the saved .pt files — PyTorch requires the class
# definition to unpickle the saved dataset objects.
# =============================================================================

from torch.utils.data import Dataset

class PressureUlcerQADataset(Dataset):
    """
    PyTorch Dataset for the pressure ulcer BioBERT QA system.
    Each item returns a dictionary of tensors representing one
    tokenised QA example.
    """
    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx]

print("PressureUlcerQADataset class defined.")

PressureUlcerQADataset class defined.


In [5]:
# =============================================================================
# Model and Data Loading
# Loads BioBERT from local cache and preprocessed datasets from Task 2.
# Model is moved to GPU immediately after loading.
# =============================================================================

# Load preprocessed datasets
train_dataset = torch.load(TRAIN_FILE, weights_only=False)
val_dataset   = torch.load(VAL_FILE,   weights_only=False)
test_dataset  = torch.load(TEST_FILE,  weights_only=False)

print(f"Datasets loaded:")
print(f"  Train:      {len(train_dataset):,} examples")
print(f"  Validation: {len(val_dataset):,} examples")
print(f"  Test:       {len(test_dataset):,} examples")

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0    # 0 workers avoids multiprocessing issues on Windows
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"\nDataLoaders created:")
print(f"  Train batches:      {len(train_loader):,}")
print(f"  Validation batches: {len(val_loader):,}")
print(f"  Test batches:       {len(test_loader):,}")

# Load BioBERT model with QA head
print(f"\nLoading BioBERT model...")
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(device)

# Count trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters()
                       if p.requires_grad)

print(f"Model loaded successfully:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Device:               {next(model.parameters()).device}")

# GPU memory after model load
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"  GPU memory used:      {allocated:.2f} GB")

Datasets loaded:
  Train:      2,365 examples
  Validation: 506 examples
  Test:       508 examples

DataLoaders created:
  Train batches:      148
  Validation batches: 32
  Test batches:       32

Loading BioBERT model...


C:\Users\MSC1\anaconda3\envs\bertqa2\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded successfully:
  Total parameters:     363,251,714
  Trainable parameters: 363,251,714
  Device:               cuda:0
  GPU memory used:      1.45 GB


## 3.5 Optimiser, Scheduler, and Training Loop

The AdamW optimiser and linear warmup scheduler are configured 
before training begins. The total number of training steps is 
calculated as the number of batches per epoch multiplied by the 
number of epochs — this value is required by the scheduler to 
compute the linear decay rate after the warmup period ends.

### Training Loop Structure

The training loop iterates over epochs. Within each epoch, every 
batch is processed as follows:

1. Input tensors are moved to the GPU device
2. The model performs a forward pass, computing start and end 
   logits for every token position in the sequence
3. The QA loss is computed as the mean of the cross-entropy 
   losses for the start and end position predictions against 
   the ground truth start and end token indices
4. Gradients are computed via backpropagation
5. Gradients are clipped to a maximum L2 norm of 1.0 to 
   prevent exploding gradients
6. The optimiser updates model weights
7. The scheduler updates the learning rate

### Validation and Checkpointing

After each training epoch, the model is evaluated on the 
validation set. Validation loss is computed in inference mode 
with gradient computation disabled, which reduces memory 
consumption and speeds up the evaluation pass. If the validation 
loss improves, the current model state is saved as the best 
checkpoint. If validation loss fails to improve for two 
consecutive epochs, early stopping halts training and the best 
checkpoint is restored.

Training loss, validation loss, and current learning rate are 
logged after each epoch to allow monitoring of the training 
trajectory.

In [6]:
# =============================================================================
# Optimiser, Scheduler, and Training Loop
# AdamW with linear warmup and decay schedule.
# Early stopping based on validation loss with patience of 2 epochs.
# Best checkpoint saved automatically when validation loss improves.
# =============================================================================

# Total training steps for scheduler
total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

print(f"Training configuration:")
print(f"  Total steps:   {total_steps:,}")
print(f"  Warmup steps:  {warmup_steps:,}")
print(f"  Steps/epoch:   {len(train_loader):,}")

# AdamW optimiser — weight decay applied to all parameters
# except bias and LayerNorm weights which should not be regularised
no_decay      = ['bias', 'LayerNorm.weight']
optim_grouped = [
    {
        'params': [p for n, p in model.named_parameters()
                   if not any(nd in n for nd in no_decay)],
        'weight_decay': WEIGHT_DECAY
    },
    {
        'params': [p for n, p in model.named_parameters()
                   if any(nd in n for nd in no_decay)],
        'weight_decay': 0.0
    }
]
optimizer = AdamW(optim_grouped, lr=LEARNING_RATE)

# Linear warmup and decay scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps    = warmup_steps,
    num_training_steps  = total_steps
)

# =============================================================================
# Training loop
# =============================================================================
best_val_loss    = float('inf')
patience_counter = 0
training_history = []

print(f"\nStarting training...")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    # ------------------------------------------------------------------
    # Training phase
    # ------------------------------------------------------------------
    model.train()
    train_loss  = 0.0
    train_steps = 0

    for batch in tqdm(train_loader,
                      desc=f"Epoch {epoch}/{NUM_EPOCHS} [Train]",
                      leave=False):

        # Move batch to GPU
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        start_positions = batch['start_position'].to(device)
        end_positions   = batch['end_position'].to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(
            input_ids       = input_ids,
            attention_mask  = attention_mask,
            token_type_ids  = token_type_ids,
            start_positions = start_positions,
            end_positions   = end_positions
        )

        loss = outputs.loss

        # Backward pass
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        # Optimiser and scheduler step
        optimizer.step()
        scheduler.step()

        train_loss  += loss.item()
        train_steps += 1

    avg_train_loss = train_loss / train_steps

    # ------------------------------------------------------------------
    # Validation phase
    # ------------------------------------------------------------------
    model.eval()
    val_loss  = 0.0
    val_steps = 0

    with torch.no_grad():
        for batch in tqdm(val_loader,
                          desc=f"Epoch {epoch}/{NUM_EPOCHS} [Val]",
                          leave=False):

            input_ids       = batch['input_ids'].to(device)
            attention_mask  = batch['attention_mask'].to(device)
            token_type_ids  = batch['token_type_ids'].to(device)
            start_positions = batch['start_position'].to(device)
            end_positions   = batch['end_position'].to(device)

            outputs = model(
                input_ids       = input_ids,
                attention_mask  = attention_mask,
                token_type_ids  = token_type_ids,
                start_positions = start_positions,
                end_positions   = end_positions
            )

            val_loss  += outputs.loss.item()
            val_steps += 1

    avg_val_loss = val_loss / val_steps
    epoch_time   = time.time() - epoch_start
    current_lr   = scheduler.get_last_lr()[0]

    # Log epoch results
    training_history.append({
        'epoch':      epoch,
        'train_loss': avg_train_loss,
        'val_loss':   avg_val_loss,
        'lr':         current_lr
    })

    print(f"Epoch {epoch}/{NUM_EPOCHS} | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"LR: {current_lr:.2e} | "
          f"Time: {epoch_time:.0f}s")

    # ------------------------------------------------------------------
    # Checkpointing and early stopping
    # ------------------------------------------------------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0

        # Save best model
        model.save_pretrained(BEST_MODEL_DIR)
        print(f"  ✅ Best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  ⚠️  No improvement. Patience: {patience_counter}/{EARLY_STOP_PAT}")

        if patience_counter >= EARLY_STOP_PAT:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

print("-" * 65)
print(f"Training complete.")
print(f"Best validation loss: {best_val_loss:.4f}")

Training configuration:
  Total steps:   444
  Warmup steps:  44
  Steps/epoch:   148

Starting training...
-----------------------------------------------------------------


Epoch 1/3 | Train Loss: 1.3378 | Val Loss: 0.8420 | LR: 1.48e-05 | Time: 2147s
  ✅ Best model saved (val_loss=0.8420)


Epoch 2/3 | Train Loss: 0.5585 | Val Loss: 0.9379 | LR: 7.40e-06 | Time: 2325s
  ⚠️  No improvement. Patience: 1/2


Epoch 3/3 | Train Loss: 0.2960 | Val Loss: 1.0124 | LR: 0.00e+00 | Time: 2260s
  ⚠️  No improvement. Patience: 2/2

Early stopping triggered at epoch 3.
-----------------------------------------------------------------
Training complete.
Best validation loss: 0.8420


## 3.6 Model Saving and Training History

The final model state is saved alongside the best checkpoint. 
While the best checkpoint at epoch 1 will be used for inference 
in Task 4, saving the final epoch model provides a complete 
record of the training run and allows comparison between the 
best and final states if required.

The training history — epoch losses and learning rates — is 
saved as a JSON file for reference in the Task 4 results 
discussion. This record documents the training trajectory 
and provides evidence that early stopping functioned correctly 
by preserving the epoch 1 checkpoint rather than the overfit 
epoch 3 state.

The tokeniser is saved alongside the model to ensure that 
inference in Task 4 uses the identical vocabulary and 
tokenisation settings as training. Loading a different 
tokeniser version at inference time would produce different 
token sequences for the same input text, causing a vocabulary 
mismatch that would degrade prediction quality.

In [7]:
# =============================================================================
# Model Saving and Training History
# Saves final model, tokeniser, and training history to disk.
# Best checkpoint was already saved during training loop.
# =============================================================================

# Save final epoch model
model.save_pretrained(MODEL_SAVE_DIR)
print(f"Final model saved to: {MODEL_SAVE_DIR}")

# Save tokeniser alongside both model checkpoints
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
tokeniser.save_pretrained(MODEL_SAVE_DIR)
tokeniser.save_pretrained(BEST_MODEL_DIR)
print(f"Tokeniser saved to:   {MODEL_SAVE_DIR}")
print(f"Tokeniser saved to:   {BEST_MODEL_DIR}")

# Save training history
history_file = "./training_history.json"
with open(history_file, 'w') as f:
    json.dump(training_history, f, indent=2)
print(f"Training history saved to: {history_file}")

# Summary
print(f"\nTraining Summary:")
print(f"{'Epoch':<8} {'Train Loss':>12} {'Val Loss':>12} {'LR':>12}")
print(f"{'-'*46}")
for h in training_history:
    marker = " ← best" if h['val_loss'] == best_val_loss else ""
    print(f"{h['epoch']:<8} {h['train_loss']:>12.4f} "
          f"{h['val_loss']:>12.4f} {h['lr']:>12.2e}{marker}")

print(f"\nBest checkpoint: Epoch 1 (val_loss={best_val_loss:.4f})")
print(f"Saved to:        {BEST_MODEL_DIR}")

Final model saved to: ./biobert_pressure_ulcer


C:\Users\MSC1\anaconda3\envs\bertqa2\lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokeniser saved to:   ./biobert_pressure_ulcer
Tokeniser saved to:   ./biobert_pressure_ulcer_best
Training history saved to: ./training_history.json

Training Summary:
Epoch      Train Loss     Val Loss           LR
----------------------------------------------
1              1.3378       0.8420     1.48e-05 ← best
2              0.5585       0.9379     7.40e-06
3              0.2960       1.0124     0.00e+00

Best checkpoint: Epoch 1 (val_loss=0.8420)
Saved to:        ./biobert_pressure_ulcer_best


## 3.7 Task 3 Summary and Critical Discussion

### Training Summary

BioBERT Large was fine-tuned on the pressure ulcer QA dataset 
for 3 epochs with early stopping. The key results are summarised 
below:

| Metric | Value |
|--------|-------|
| Base model | dmis-lab/biobert-large-cased-v1.1-squad |
| Total parameters | 363,251,714 |
| Training examples | 2,365 |
| Validation examples | 506 |
| Best epoch | 1 |
| Best validation loss | 0.8420 |
| Final training loss | 0.2960 |
| Training time per epoch | ~36 minutes |
| Hardware | NVIDIA RTX 3090 25.8 GB |

### Interpretation of Training Trajectory

The training trajectory — decreasing train loss alongside 
increasing validation loss after epoch 1 — is a textbook 
overfitting signature. Train loss decreased consistently 
from 1.3378 at epoch 1 to 0.2960 at epoch 3, while 
validation loss increased from 0.8420 to 1.0124 over the 
same period. This divergence indicates that the model was 
memorising training examples rather than learning 
generalisable clinical QA patterns by epoch 2.

Early stopping correctly identified epoch 1 as the optimal 
checkpoint and preserved the best model state automatically. 
The epoch 1 checkpoint is the model that will be loaded for 
inference and evaluation in Task 4.

### Why Overfitting Occurred at Epoch 1

The early onset of overfitting at epoch 1 is attributable 
to the relatively small training corpus of 2,365 examples 
in combination with the large model capacity of BioBERT 
Large at 363 million parameters. The parameter-to-example 
ratio of approximately 153,000 parameters per training 
example is extremely high by supervised learning standards 
— the model has far more representational capacity than 
the training data can fully constrain. This imbalance 
allows the model to fit training examples precisely without 
learning patterns that generalise beyond them.

It is worth noting that the epoch 1 validation loss of 
0.8420 represents a substantial improvement over the 
untrained QA head, which would produce near-random span 
predictions. The model has learned meaningful clinical QA 
patterns — the overfitting concern is relative rather than 
absolute.

### Limitations and Critical Discussion

**Dataset size.** The primary limitation of this training 
run is the training corpus size of 2,365 examples. BioBERT 
Large with 363 million parameters requires substantially 
more training data to reach its full potential. The original 
BioBERT paper fine-tuned on datasets of 87,000 to 100,000 
examples for QA tasks. A dataset of this scale would be 
expected to produce significantly lower validation loss and 
better generalisation. The dataset size is ultimately constrained by the size of the knowledge base corpus — 2,109 chunks at 5 pairs per chunk with a retention rate of approximately 58% produces a natural ceiling of approximately 6,000 pairs before deduplication. Expanding the dataset would require either a substantially larger document corpus or a reduction in quality filtering stringency, both of which carry trade-offs in clinical relevance.

**Single epoch convergence.** The fact that the best 
checkpoint is at epoch 1 raises the question of whether 
the learning rate is too high, causing the model to 
overshoot the optimal parameter configuration within a 
single epoch. A lower learning rate such as 1e-5 combined 
with more training examples would likely produce a smoother 
training trajectory with improvement across more epochs. 
This represents a hyperparameter configuration that could 
be explored in future work.

**No-answer detection.** The training loss reported is the 
combined span prediction loss for answerable examples and 
the CLS position loss for unanswerable examples. The dataset 
contains only 232 unanswerable training examples (9.8%), 
which may result in underfitting of the no-answer detection 
capability relative to the span prediction capability. The 
confidence threshold applied at inference time in Task 4 
partially compensates for this imbalance.

**Evaluation metric.** Validation loss as the early stopping 
criterion measures the model's cross-entropy loss on the 
tokenised span positions rather than the quality of the 
extracted answer spans as strings. A model with low 
validation loss does not necessarily produce the highest 
BERTScore — these are related but not identical objectives. 
Task 4 provides the definitive assessment of model quality 
through BERTScore evaluation on the held-out test set.

In [8]:
# =============================================================================
# Task 3 Complete — Final File Verification
# =============================================================================

import os

output_dirs = {
    "Best model checkpoint": BEST_MODEL_DIR,
    "Final model checkpoint": MODEL_SAVE_DIR,
}

output_files = {
    "Training history": "./training_history.json"
}

print("Task 3 Complete — Output Verification:")
print("-" * 55)

for name, dirpath in output_dirs.items():
    if os.path.exists(dirpath):
        files    = os.listdir(dirpath)
        size_mb  = sum(
            os.path.getsize(os.path.join(dirpath, f))
            for f in files
        ) / 1024 / 1024
        print(f"  ✅ {name:<28} ({size_mb:.0f} MB, {len(files)} files)")
    else:
        print(f"  ❌ {name:<28} MISSING")

for name, filepath in output_files.items():
    if os.path.exists(filepath):
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  ✅ {name:<28} ({size_kb:.1f} KB)")
    else:
        print(f"  ❌ {name:<28} MISSING")

print("-" * 55)
print(f"\nBest model ready for Task 4 inference.")
print(f"Load from: {BEST_MODEL_DIR}")

Task 3 Complete — Output Verification:
-------------------------------------------------------
  ✅ Best model checkpoint        (1388 MB, 6 files)
  ✅ Final model checkpoint       (1388 MB, 6 files)
  ✅ Training history             (0.4 KB)
-------------------------------------------------------

Best model ready for Task 4 inference.
Load from: ./biobert_pressure_ulcer_best


### Task 3 completed